In [3]:
from __future__ import annotations

import copy
import json
from itertools import product
from pathlib import Path


# ============================================================
# Paths
# ============================================================

ROOT_DIR = Path("/home/blue2959/monotonic_tts/exp_configs/v2.0/grid_raw_configs")
TEMPLATE_DIR = ROOT_DIR / "conv-RF=1_text-RF=1"

TEMPLATE_DATA_CONFIG = TEMPLATE_DIR / "data_config.json"
TEMPLATE_MODEL_CONFIG = TEMPLATE_DIR / "model_config.json"


# ============================================================
# Grid settings
# ============================================================

text_rf_values = [1, 3, 5, 7, 9]
conv_rf_values = [1, 3, 5, 7, 9, 11, 13, 15]
overwrite = True


# ============================================================
# JSON utilities
# ============================================================

def load_json(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Template JSON not found: {path}"
        )

    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def save_json(data: dict, path: Path) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            indent=4,
            ensure_ascii=False,
        )
        f.write("\n")


# ============================================================
# RF conversion
# ============================================================

def validate_rf(rf: int) -> None:
    if rf < 1 or rf % 2 == 0:
        raise ValueError(
            f"RF must be a positive odd integer, got {rf}."
        )


def make_text_kernel_sizes(rf: int) -> list[int]:
    """
    RF=1 -> [1]
    RF=3 -> [3]
    RF=5 -> [3, 3]
    RF=7 -> [3, 3, 3]
    RF=9 -> [3, 3, 3, 3]
    """
    validate_rf(rf)

    if rf == 1:
        return [1]

    num_layers = (rf - 1) // 2
    return [3] * num_layers


def make_conv_settings(
    rf: int,
) -> tuple[int, list[int]]:
    """
    Returns:
        conv_num_layers
        conv_kernel_size

    RF=1 -> 1 layer  with 1x1 kernel
    RF=3 -> 1 layer  with 3x3 kernel
    RF=5 -> 2 layers with 3x3 kernel
    RF=7 -> 3 layers with 3x3 kernel
    RF=9 -> 4 layers with 3x3 kernel
    """
    validate_rf(rf)

    if rf == 1:
        return 1, [1, 1]

    num_layers = (rf - 1) // 2
    return num_layers, [3, 3]


# ============================================================
# Load templates
# ============================================================

base_data_config = load_json(
    TEMPLATE_DATA_CONFIG
)

base_model_config = load_json(
    TEMPLATE_MODEL_CONFIG
)


# ============================================================
# Generate configs
# ============================================================

created_count = 0
skipped_count = 0

for conv_rf, text_rf in product(
    conv_rf_values,
    text_rf_values,
):
    experiment_name = (
        f"conv-RF={conv_rf}_text-RF={text_rf}"
    )

    experiment_dir = ROOT_DIR / experiment_name
    config_dir = experiment_dir

    data_config_path = (
        config_dir / "data_config.json"
    )
    model_config_path = (
        config_dir / "model_config.json"
    )

    if (
        not overwrite
        and data_config_path.exists()
        and model_config_path.exists()
    ):
        print(
            f"[SKIP] Config already exists: "
            f"{config_dir}"
        )
        skipped_count += 1
        continue

    data_config = copy.deepcopy(
        base_data_config
    )
    model_config = copy.deepcopy(
        base_model_config
    )

    # ========================================================
    # data_config.json
    # ========================================================

    data_config["extra_exp"]["exp_name"] = (
        f"v2.0_vctk_conv-RF={conv_rf}"
        f"+text-RF={text_rf}"
    )

    data_config["extra_exp"]["exp_variant"] = (
        "v2.0_vctk+rf_grid_search"
    )

    data_config["extra_exp"]["base_dir"] = (
        "/shared/data_zfs/blue2959/"
        "ND_Aligner/experiments/v2.0/rf_grid"
    )

    # ========================================================
    # model_config.json: text encoder
    # ========================================================

    text_kernel_sizes = make_text_kernel_sizes(
        text_rf
    )

    model_config["nd_aligner"]["txt_enc"][
        "kernel_sizes"
    ] = text_kernel_sizes

    # ========================================================
    # model_config.json: contextual Conv2d scorer
    # ========================================================

    conv_num_layers, conv_kernel_size = (
        make_conv_settings(conv_rf)
    )

    aligner_config = model_config[
        "nd_aligner"
    ]["aligner"]

    aligner_config["conv_num_layers"] = (
        conv_num_layers
    )

    aligner_config["conv_kernel_size"] = (
        conv_kernel_size
    )

    # ========================================================
    # Save
    # ========================================================

    save_json(
        data_config,
        data_config_path,
    )

    save_json(
        model_config,
        model_config_path,
    )

    print(
        f"[CREATE] {config_dir}\n"
        f"         text RF={text_rf}: "
        f"kernel_sizes={text_kernel_sizes}\n"
        f"         conv RF={conv_rf}: "
        f"num_layers={conv_num_layers}, "
        f"kernel={conv_kernel_size}\n"
        f"         exp_name="
        f"{data_config['extra_exp']['exp_name']}"
    )

    created_count += 1


print()
print(f"Created: {created_count}")
print(f"Skipped: {skipped_count}")
print(f"Root: {ROOT_DIR.resolve()}")

[CREATE] /home/blue2959/monotonic_tts/exp_configs/v2.0/grid_raw_configs/conv-RF=1_text-RF=1
         text RF=1: kernel_sizes=[1]
         conv RF=1: num_layers=1, kernel=[1, 1]
         exp_name=v2.0_vctk_conv-RF=1+text-RF=1
[CREATE] /home/blue2959/monotonic_tts/exp_configs/v2.0/grid_raw_configs/conv-RF=1_text-RF=3
         text RF=3: kernel_sizes=[3]
         conv RF=1: num_layers=1, kernel=[1, 1]
         exp_name=v2.0_vctk_conv-RF=1+text-RF=3
[CREATE] /home/blue2959/monotonic_tts/exp_configs/v2.0/grid_raw_configs/conv-RF=1_text-RF=5
         text RF=5: kernel_sizes=[3, 3]
         conv RF=1: num_layers=1, kernel=[1, 1]
         exp_name=v2.0_vctk_conv-RF=1+text-RF=5
[CREATE] /home/blue2959/monotonic_tts/exp_configs/v2.0/grid_raw_configs/conv-RF=1_text-RF=7
         text RF=7: kernel_sizes=[3, 3, 3]
         conv RF=1: num_layers=1, kernel=[1, 1]
         exp_name=v2.0_vctk_conv-RF=1+text-RF=7
[CREATE] /home/blue2959/monotonic_tts/exp_configs/v2.0/grid_raw_configs/conv-RF=1_text-RF=9